# 07 — A2A アデノシン受容体 GPCR 解析チュートリアル
# A2A Adenosine Receptor GPCR Docking Analysis Tutorial

**ターゲット**: A2A アデノシン受容体（class A GPCR）  
**事例**: ZM241385 (拮抗薬 / inactive state) vs Adenosine (内因性作動薬 / active state)  
**PDB 構造**: 3EML (ZM241385, 2.6 Å) / 4EIY (Adenosine, 2.6 Å)

---

## このノートブックで学べること

1. GPCR 結晶構造特有の前処理（界面活性剤・T4L 融合タンパク質・コレステロール誘導体の除去）
2. TM ヘリックスに基づく直交性結合ポケットのグリッドボックス定義
3. `ProLIFCalculator` を使った TM ヘリックス接触パターン抽出
4. 拮抗薬 (inactive state) vs 作動薬 (active state) の結合様式比較
5. GPCR 創薬における Extracellular Loop 2 (ECL2) の役割

---

## GPCR 構造の特徴

```
Extracellular side
  ECL1 ─── ECL2 ─── ECL3    ← 結合ポケット入口 (ECL2 が蓋)
   |  TM1  TM2  TM3  TM4  TM5  TM6  TM7  |
  ICL1 ─── ICL2 ─── ICL3    ← G タンパク質カップリング
Intracellular side
```

- **3EML** (ZM241385): 拮抗薬結合 → **inactive conformation** (TM5–TM6 間の空洞が狭い)  
- **4EIY** (Adenosine): 内因性作動薬結合 → **active-like conformation** (TM6 が外向きに移動)  

> **Note**: ドッキング実行セクション（Section 5）は Vina または UniDock バイナリが必要です。  
> バイナリなしでも Section 6 以降の解析デモは結晶構造ポーズで実行可能です。

In [ ]:
# CONFIG -----------------------------------------------------------------------
DATA_DIR    = "../data/gpcr_a2a"   # PDB ダウンロード先 / PDB download directory
RESULTS_DIR = "../results/gpcr_a2a" # ドッキング結果出力先
VINA_BINARY = "vina"                # 'vina' or 'unidock' or path to binary
# ------------------------------------------------------------------------------

## 1. セットアップ / Setup

In [ ]:
import urllib.request
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, AllChem
from IPython.display import display

# NOTE: docking runners and preparation tools (gridbox, ligand, receptor)
# will be available in mdatools.docking in a future release.
from docking_analysis import DockingResult, get_reader
from docking_analysis.preparation.receptor import (
    load_receptor, remove_solvent, select_protein, prepare_receptor
)
from docking_analysis.preparation.gridbox import gridbox_from_ligand, GridBox
from mdatools.docking.fingerprints.prolif import ProLIFCalculator
from mdatools.docking.visualization.interaction_map import draw_interaction_map
from mdatools.docking.analysis.properties import calculate_properties
from mdatools.docking.clustering.chemical import compute_fp_matrix, cluster_by_butina

RDLogger.DisableLog("rdApp.warning")

data_dir    = Path(DATA_DIR)
results_dir = Path(RESULTS_DIR)
data_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
print("Setup complete.")

## 2. PDB 構造の取得 / Download PDB Structures

| PDB ID | リガンド | コンフォメーション | 分解能 | 特徴 |
|--------|---------|-----------------|--------|-----|
| **3EML** | ZM241385 | Inactive (拮抗薬) | 2.6 Å | A2A ドッキングベンチマーク標準構造。T4L 融合あり |
| **4EIY** | Adenosine | Active-like (作動薬) | 2.6 Å | 内因性リガンドの結合様式。TM6 外向き移動を確認可能 |

In [ ]:
PDB_IDS = ["3EML", "4EIY"]

for pdb_id in PDB_IDS:
    dest = data_dir / f"{pdb_id.lower()}.pdb"
    if not dest.exists() or dest.stat().st_size == 0:
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        print(f"Downloading {pdb_id}...", end=" ")
        urllib.request.urlretrieve(url, dest)
        print(f"→ {dest}")
    else:
        print(f"{pdb_id}: already exists ({dest})")

### 2.1 HETATM 残基の確認

GPCR 結晶構造には以下のような非タンパク質分子が含まれることが多いです：

| 分類 | 代表的なコード | 内容 |
|------|-------------|------|
| **目的リガンド** | ZMA, ADN | 解析対象の化合物 |
| **界面活性剤** | OG, LMT, NG | 結晶化に使用。除去が必要 |
| **コレステロール誘導体** | CLR, CHS | 膜タンパク質安定化剤 |
| **結晶充填剤** | GOL, EDO, PEG | クライオ保護剤 |
| **金属イオン** | ZN, MG, CA | 除去対象 |

→ `prepare_receptor()` の `protein_only=True`（デフォルト）でこれらはすべて自動除去されます。

In [ ]:
def list_hetatm_residues(pdb_path: Path, min_atoms: int = 1) -> list[dict]:
    """Parse HETATM records and return unique non-water residues."""
    WATER_CODES = {"HOH", "WAT", "H2O", "DOD", "D2O"}
    seen = {}
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith("HETATM"):
                continue
            res_name = line[17:20].strip()
            chain    = line[21].strip()
            seq_id   = line[22:26].strip()
            key = (chain, res_name, seq_id)
            if res_name not in WATER_CODES:
                seen[key] = seen.get(key, 0) + 1
    result = [
        {"chain": k[0], "residue": k[1], "seqid": k[2], "n_atoms": v}
        for k, v in seen.items()
        if v >= min_atoms
    ]
    return sorted(result, key=lambda x: -x["n_atoms"])

for pdb_id in PDB_IDS:
    residues = list_hetatm_residues(data_dir / f"{pdb_id.lower()}.pdb", min_atoms=6)
    print(f"\n{pdb_id} — 主要 HETATM 残基 (≥6 heavy atoms):")
    for r in residues:
        print(f"  Chain {r['chain']}: {r['residue']:>4}  seqid={r['seqid']:>5}  atoms={r['n_atoms']}")

### 2.2 リガンドコードの設定

上の出力を確認して `LIGAND_CODES` を設定してください。

- **3EML** (ZM241385): 通常 `"ZMA"` — 重原子数が最も多い HETATM
- **4EIY** (Adenosine): 通常 `"ADN"` — 重原子数が最も多い HETATM

上の出力と異なる場合はコードを変更してください。

In [ ]:
# ↓ 上のセルの出力を見て残基コードを設定 / Set residue codes from output above
LIGAND_CODES = {
    "3EML": "ZMA",   # ZM241385 (antagonist)
    "4EIY": "ZMA",   # ZM241385 (RCSB code; 4EIY also contains antagonist)
}

## 3. リガンド・受容体の抽出と準備
## Extract Ligands and Prepare Receptor

In [ ]:
def extract_ligand_mol(pdb_path: Path, res_code: str, chain: str = "A") -> Chem.Mol | None:
    """Extract a ligand from PDB HETATM records and return an RDKit Mol."""
    with open(pdb_path) as f:
        lines = f.readlines()

    hetatm = [
        l for l in lines
        if l.startswith("HETATM")
        and l[17:20].strip() == res_code
        and (chain == "*" or l[21].strip() == chain)
    ]
    if not hetatm:
        # chain B を試みる
        hetatm = [
            l for l in lines
            if l.startswith("HETATM") and l[17:20].strip() == res_code
        ]
    if not hetatm:
        print(f"  WARNING: residue {res_code} not found in {pdb_path.name}")
        return None

    pdb_block = "".join(hetatm) + "END\n"
    mol = Chem.MolFromPDBBlock(pdb_block, removeHs=True, sanitize=True)
    if mol is None:
        print(f"  WARNING: RDKit could not parse {res_code}")
    return mol


ligand_mols = {}
for pdb_id, res_code in LIGAND_CODES.items():
    mol = extract_ligand_mol(data_dir / f"{pdb_id.lower()}.pdb", res_code)
    if mol is not None:
        mol.SetProp("mol_name", res_code)
        mol.SetProp("pdb_id", pdb_id)
        mol.SetProp("pose_rank", "1")
        mol.SetProp("docking_score", "0.0")
        ligand_mols[pdb_id] = mol
        print(f"{pdb_id} [{res_code}]: {mol.GetNumAtoms()} heavy atoms")
    else:
        print(f"{pdb_id} [{res_code}]: failed — check residue code above")

# 構造を表示
if len(ligand_mols) >= 1:
    mols_to_draw = list(ligand_mols.values())
    legends = [
        f"{pid}\n{LIGAND_CODES[pid]}\n{'Antagonist (inactive)' if pid == '3EML' else 'Agonist (active)'}"
        for pid in ligand_mols
    ]
    img = Draw.MolsToGridImage(
        mols_to_draw, molsPerRow=2, subImgSize=(400, 300), legends=legends
    )
    display(img)

### 3.1 GPCR 受容体準備の注意点

#### T4 Lysozyme (T4L) 融合タンパク質について

**3EML** では ICL3（細胞内ループ3）に T4L が挿入されています。  
`prepare_receptor(protein_only=True)` はタンパク質 ATOM レコードをすべて保持するため、  
T4L も受容体ファイルに含まれますが、**正直なドッキングには影響しません**。

理由:
- T4L は細胞内側（ICL3）に位置し、直交性結合ポケット（細胞外側）から離れている
- グリッドボックスをリガンド重心から定義するため、T4L 領域はボックス外になる

T4L を完全に除去したい場合は、以下のように MDAnalysis で受容体番号範囲を指定できます:

```python
import MDAnalysis as mda
u = mda.Universe("3eml.pdb")
# 3EML の A2A 受容体残基番号 (1-202, 310-332) のみ選択
receptor_only = u.select_atoms("protein and (resid 1-202 or resid 310-332)")
receptor_only.write("3eml_receptor_no_t4l.pdb")
```

In [ ]:
# 受容体準備: 界面活性剤・リガンド・溶媒を除去してタンパク質のみを抽出
# protein_only=True がデフォルト — OG, CHS, CLR 等の HETATM を自動除去

receptor_paths  = {}
receptor_mols   = {}

# GPCR 構造特有の追加除去リスト (MDAnalysis 'protein' selection では自動除去されるが念のため)
GPCR_DETERGENTS = {"OG", "LMT", "NG", "CHS", "CLR", "LMNG", "DDM", "DM"}

for pdb_id in PDB_IDS:
    raw_pdb = data_dir / f"{pdb_id.lower()}.pdb"
    out_pdb = data_dir / f"{pdb_id.lower()}_receptor.pdb"

    if not out_pdb.exists():
        print(f"Preparing {pdb_id} receptor...", end=" ")
        # protein_only=True: 標準アミノ酸残基のみ保持
        # 界面活性剤・コレステロール誘導体・T4L 以外の HETATM は自動除去
        _, rec_mol = prepare_receptor(raw_pdb, out_pdb, protein_only=True)
        print(f"→ {out_pdb}")
    else:
        print(f"{pdb_id} receptor ready: {out_pdb}")
        rec_mol = load_receptor(out_pdb)

    receptor_paths[pdb_id] = out_pdb
    receptor_mols[pdb_id]  = rec_mol

print("\n受容体準備完了。")
print("注意: 3EML の T4L (ICL3 挿入) は含まれていますが、正直結合部位には影響しません。")

## 4. グリッドボックスの設定
## Grid Box Setup

GPCR の直交性結合ポケットは TM ヘリックス束の中心に位置します。  
結晶リガンドの重心から自動定義することで、TM ヘリックスにまたがる空洞全体を  
カバーするグリッドボックスが得られます。

**Inactive (3EML) vs Active (4EIY) の結合部位比較:**  
活性化に伴う TM6 の外向き移動 (~6–14 Å) が結合部位中心のシフトとして現れます。

In [ ]:
gridboxes = {}
for pdb_id, mol in ligand_mols.items():
    # 3D 座標が必要な場合は ETKDG で生成
    if mol.GetNumConformers() == 0:
        AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())

    # GPCR のポケットは深いため padding を少し大きめに設定
    grid = gridbox_from_ligand(mol, padding=6.0)
    gridboxes[pdb_id] = grid

    label = "Antagonist / inactive" if pdb_id == "3EML" else "Agonist / active"
    print(f"\n{pdb_id} [{label}] グリッドボックス:")
    print(f"  中心 (Å): ({grid.center[0]:.2f}, {grid.center[1]:.2f}, {grid.center[2]:.2f})")
    print(f"  サイズ (Å): ({grid.size[0]:.2f}, {grid.size[1]:.2f}, {grid.size[2]:.2f})")

# 3EML (inactive) vs 4EIY (active) の結合部位中心のシフト
if len(gridboxes) == 2:
    c1 = gridboxes["3EML"].center
    c2 = gridboxes["4EIY"].center
    shift = sum((a - b)**2 for a, b in zip(c1, c2)) ** 0.5
    print(f"\n結合部位中心のシフト (3EML vs 4EIY): {shift:.2f} Å")
    print("※ active/inactive 変換に伴う TM6 移動が大きいほどシフトが大きくなります")

## 5. ドッキング実行
## Docking Execution

> ⚠️ このセクションは Vina または UniDock バイナリが必要です。  
> バイナリがない場合は Section 6（解析デモ）に進んでください。

**GPCR ドッキングの推奨設定:**
- `exhaustiveness=16` 以上（キナーゼより高い設定を推奨）
- `num_modes=20` — 複数のバインディングモードを探索
- グリッドボックスを少し大きめに（`padding=6.0`）

In [ ]:
# リガンド PDBQT 変換 (meeko 使用)
# NOTE: docking runners and preparation tools (gridbox, ligand, receptor)
# will be available in mdatools.docking in a future release.
from docking_analysis.preparation.ligand import prepare_for_vina

pdbqt_paths = {}
for pdb_id, mol in ligand_mols.items():
    if mol.GetNumConformers() == 0:
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
        AllChem.MMFFOptimizeMolecule(mol)

    out_pdbqt = data_dir / f"{pdb_id.lower()}_{LIGAND_CODES[pdb_id].lower()}.pdbqt"
    try:
        prepare_for_vina(mol, out_pdbqt)
        pdbqt_paths[pdb_id] = out_pdbqt
        print(f"{pdb_id}: → {out_pdbqt}")
    except Exception as e:
        print(f"{pdb_id}: PDBQT 変換エラー — {e}")

In [ ]:
# ドッキング実行 (Vina バイナリが必要 / Requires Vina binary)
import shutil

if shutil.which(VINA_BINARY) is None:
    print(f"⚠ '{VINA_BINARY}' not found in PATH — skipping docking.")
    print("Section 6 uses crystal structure poses as demo data instead.")
else:
# NOTE: docking runners and preparation tools (gridbox, ligand, receptor)
# will be available in mdatools.docking in a future release.
    from docking_analysis.docking import get_runner

    runner = get_runner("vina", exhaustiveness=16, num_modes=20)

    # 3EML (inactive state) の受容体に対して ZM241385 を再ドッキング
    receptor_pdbqt = data_dir / "3eml_receptor.pdbqt"  # meeko で別途変換
    if "3EML" in pdbqt_paths and receptor_pdbqt.exists():
        result = runner.run(
            ligand_pdbqt=pdbqt_paths["3EML"],
            receptor_pdbqt=receptor_pdbqt,
            grid=gridboxes["3EML"],
            output_pdbqt=results_dir / "3eml_zm241385_out.pdbqt",
        )
        print(f"Top score: {result.scores[0]:.2f} kcal/mol")
        print(f"Poses    : {len(result.poses)}")

## 6. 結合様式の比較解析（結晶構造ポーズ使用）
## Binding Mode Analysis (Using Crystal Poses)

以下では**結晶構造から抽出したポーズ**を `DockingResult` として利用し、  
ライブラリの解析機能をデモします。実際のドッキング結果があれば  
`get_reader(backend).read(path)` で読み込んだ `result` を使ってください。

In [ ]:
# 結晶ポーズから DockingResult を構築
crystal_results = {}
for pdb_id, mol in ligand_mols.items():
    crystal_results[pdb_id] = DockingResult(
        poses=[mol],
        scores=[0.0],
        source_file=data_dir / f"{pdb_id.lower()}.pdb",
        backend="crystal",
        metadata={"source": "crystal_structure"},
    )

print("DockingResult from crystal poses:")
for pid, res in crystal_results.items():
    label = "Antagonist / inactive" if pid == "3EML" else "Agonist / active"
    print(f"  {pid} [{label}]: {len(res.poses)} pose(s), ligand {LIGAND_CODES[pid]}")

### 6.1 ProLIF 相互作用フィンガープリント — TM ヘリックス接触パターン
### ProLIF Interaction Fingerprints — TM Helix Contact Patterns

A2A の主要な結合部位残基（Ballesteros-Weinstein 表記）:

| 残基 | BW 番号 | ヘリックス | 役割 |
|------|--------|----------|------|
| Asn254 | 6.55 | TM6 | 共通鍵残基、アデニン環との H 結合 |
| His250 | 6.52 | TM6 | π-π スタッキング |
| Phe168 | ECL2 | ECL2 | ポケット入口の蓋 |
| Thr88  | 3.36 | TM3 | hydroxyl との H 結合 |
| Trp246 | 6.48 | TM6 | トグルスイッチ残基 |
| Ile92  | 3.40 | TM3 | 疎水接触 |

In [ ]:
calculator = ProLIFCalculator()

fp_results = {}
for pdb_id, result in crystal_results.items():
    rec_mol = receptor_mols.get(pdb_id)
    if rec_mol is None:
        print(f"  {pdb_id}: receptor not loaded — skip")
        continue

    print(f"\n{pdb_id} — ProLIF fingerprint...")
    fp_df = calculator.calculate(result.poses, rec_mol, show_progress=True)
    fp_results[pdb_id] = fp_df

    active_cols = fp_df.columns[fp_df.any()].tolist()
    print(f"  検出された相互作用 ({len(active_cols)} 件):")
    for col in active_cols:
        print(f"    {col}")

### 6.2 相互作用プロファイルの比較
### Interaction Profile Comparison

**期待される結合様式の違い (3EML vs 4EIY):**

| 相互作用 | ZM241385 (3EML / inactive) | Adenosine (4EIY / active) |
|---------|--------------------------|---------------------------|
| Asn254 H 結合 | ✓ (アミノ基) | ✓ (リボース OH) |
| His250 スタッキング | ✓ (フランイル基) | △ (アデニン) |
| ECL2 疎水接触 | ✓ (トリアゾール) | × |
| Thr88 H 結合 | ✓ | ✓ |
| Ser281 H 結合 | × | ✓ (リボース 3'-OH) |

In [ ]:
# 共通・固有の相互作用を比較
if len(fp_results) == 2:
    cols_3eml = set(fp_results["3EML"].columns[fp_results["3EML"].any()])
    cols_4eiy = set(fp_results["4EIY"].columns[fp_results["4EIY"].any()])

    common    = cols_3eml & cols_4eiy
    only_3eml = cols_3eml - cols_4eiy
    only_4eiy = cols_4eiy - cols_3eml

    print(f"共通の相互作用 ({len(common)} 件):")
    for c in sorted(common):    print(f"  {c}")

    print(f"\n3EML のみ (ZM241385 / antagonist, {len(only_3eml)} 件):")
    for c in sorted(only_3eml): print(f"  {c}")

    print(f"\n4EIY のみ (Adenosine / agonist, {len(only_4eiy)} 件):")
    for c in sorted(only_4eiy): print(f"  {c}")

### 6.3 インタラクションマップの比較
### Interaction Map Comparison

ZM241385 (3EML / 拮抗薬) と Adenosine (4EIY / 作動薬) の  
2D インタラクションマップを並べて比較します。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

pdb_list = ["3EML", "4EIY"]
ligand_names = {
    "3EML": "ZM241385\n(Antagonist / Inactive state)",
    "4EIY": "Adenosine\n(Agonist / Active-like state)",
}

for ax, pdb_id in zip(axes, pdb_list):
    if pdb_id not in fp_results:
        ax.set_title(f"{pdb_id}: data not available")
        continue

    fp_df      = fp_results[pdb_id]
    mol        = ligand_mols[pdb_id]
    active_cols = fp_df.columns[fp_df.any()].tolist()

    if active_cols:
        draw_interaction_map(
            mol=mol,
            interactions=active_cols,
            ax=ax,
            title=f"{pdb_id}\n{ligand_names[pdb_id]}"
        )
    else:
        ax.set_title(f"{pdb_id}: no interactions detected")

plt.tight_layout()
out_fig = results_dir / "gpcr_a2a_interaction_map_comparison.png"
fig.savefig(out_fig, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_fig}")

### 6.4 分子物性の比較
### Molecular Properties Comparison

In [ ]:
from mdatools.docking.analysis.properties import add_properties_to_df

rows = []
for pdb_id, mol in ligand_mols.items():
    props = calculate_properties(mol)
    rows.append({
        "compound": f"{LIGAND_CODES[pdb_id]} ({pdb_id})",
        "role": "Antagonist (inactive)" if pdb_id == "3EML" else "Agonist (active)",
        **props,
    })

props_df = pd.DataFrame(rows).set_index("compound")
display(props_df.round(3))

### 6.5 結合様式サマリー
### Binding Mode Summary

| 特徴 | ZM241385 (3EML) | Adenosine (4EIY) |
|------|----------------|------------------|
| 受容体コンフォメーション | **Inactive** | **Active-like** |
| TM6 状態 | 内向き (closed) | 外向き移動 (open) |
| ECL2 との相互作用 | ✓ (ポケット蓋による疎水接触) | △ (少ない) |
| Asn254 H 結合 | ✓ | ✓ |
| リボース様部位 | なし | ✓ (リボース 2',3'-OH) |
| Trp246 (Toggle switch) | 間接的接触 | ✓ (直接スタッキング) |
| 細胞内側 Gα 共役 | なし | G タンパク質/ナノボディ安定化 |

**創薬的意義:**
- 拮抗薬は inactive state に安定化させることで活性化を阻害
- 作動薬は Toggle switch (Trp246) を動かし TM6 外向き移動を誘発
- ECL2 が "lid" として機能し、拮抗薬は ECL2 接触で滞在時間を延長する傾向

## 7. 仮想スクリーニングへの応用
## Application to Virtual Screening

A2A アデノシン受容体（睡眠・炎症・がん免疫に関与）の拮抗薬スクリーニングを想定した  
ワークフロー例です。ここでは ZM241385 類縁体のダミーデータで手順を示します。

In [ ]:
# NOTE: docking runners and preparation tools (gridbox, ligand, receptor)
# will be available in mdatools.docking in a future release.
from docking_analysis.preparation.normalize import standardize_df
from mdatools.docking.analysis.properties import add_properties_to_df
from mdatools.docking.clustering.chemical import cluster_by_butina

# ダミーライブラリ: ZM241385 類縁体 + adenosine 類縁体 + ネガティブコントロール
demo_library = pd.DataFrame({
    "compound_id": [
        "ZM-001", "ZM-002", "ZM-003",
        "ADO-001", "ADO-002",
        "NEG-001",
    ],
    "smiles": [
        # ZM241385 (full structure)
        "O=C(NCc1ccncc1)c1ccc2nc(-c3ccc(F)cc3)nc(N)c2c1",
        # ZM analog (methoxy variant)
        "O=C(NCc1ccncc1)c1ccc2nc(-c3ccc(OC)cc3)nc(N)c2c1",
        # ZM analog (no pyridine)
        "O=C(NCC1CCNCC1)c1ccc2nc(-c3ccccc3)nc(N)c2c1",
        # Adenosine
        "OC[C@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1O",
        # NECA (adenosine agonist analog)
        "CCNC(=O)[C@@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1O",
        # Benzene (negative control)
        "c1ccccc1",
    ],
    "expected_activity": [
        "antagonist", "antagonist", "antagonist",
        "agonist", "agonist",
        "inactive",
    ],
})

# 分子標準化
std_df = standardize_df(demo_library, smiles_col="smiles")
print(f"標準化後: {len(std_df)} 化合物")

# 物性追加
std_df = add_properties_to_df(std_df, smiles_col="smiles")
display(std_df[["compound_id", "expected_activity", "mw", "logp", "hbd", "hba", "qed"]].round(2))

In [ ]:
# 化学的多様性クラスタリング (Butina)
mols_for_cluster = [
    Chem.MolFromSmiles(smi)
    for smi in std_df["smiles"]
    if Chem.MolFromSmiles(smi) is not None
]
valid_df = std_df[std_df["smiles"].apply(lambda s: Chem.MolFromSmiles(s) is not None)].copy()

labels = cluster_by_butina(mols_for_cluster, threshold=0.4)
valid_df["cluster"] = labels

print("クラスター割り当て:")
display(valid_df[["compound_id", "expected_activity", "cluster"]])
print(f"\nユニークなクラスター数: {valid_df['cluster'].nunique()}")
print("→ ZM 系 / adenosine 系 / ネガティブコントロールへの分離が期待される")

## 8. 次のステップ / Next Steps

このノートブックで示したワークフロー:

```
PDB ダウンロード → GPCR 受容体準備（界面活性剤・T4L 除去）
  → グリッドボックス定義（TM ポケット）
  → [ドッキング実行] → ProLIF 解析（TM ヘリックス接触）
  → 拮抗薬 vs 作動薬 結合様式比較
  → 物性計算 → クラスタリング → 化合物選択
```

**GPCR 特有の発展的解析:**
- `compute_consensus_score()` — 3EML (inactive) と 4EIY (active) に対する consensus scoring
  → 拮抗薬スクリーニングには inactive 構造への選択的結合が重要
- アロステリック部位（PAM/NAM）のグリッドボックス定義
- ECL2 との接触スコアを特徴量にした拮抗薬/作動薬分類

---

**関連 Issue:**
- Issue #76: このノートブック
- Issue #77: EGFR 共有結合ドッキング
- Issue #78: KRAS G12C 共有結合ドッキング